# 🎲 00 · 马尔可夫决策过程 MDP 与贝尔曼方程（强化学习地基）

> 面试定位：MDP 五元组、价值函数、贝尔曼期望/最优方程是所有 RL 岗位的第一问。
> 本篇以**手写 numpy** 实现一个 4×4 网格世界，把每条公式变成可见的数字。
> 标记：🔴 必背 / 🟡 掌握 / 🟢 了解。

---

## 1. 强化学习是什么（🔴 与其他 ML 的区别）

### 一句话定义

**强化学习（Reinforcement Learning, RL）** 是智能体（Agent）在**环境（Environment）**中通过**试错**学习"在什么状态做什么动作能获得最多**累积奖励**"的范式。

### 与监督/无监督学习对比（背）

| 维度 | 监督学习 | 无监督学习 | 强化学习 |
|---|---|---|---|
| 数据来源 | 标注样本 (x, y) | 无标注样本 | **与环境交互产生的轨迹** |
| 学习信号 | 标签 y | 结构/分布 | **标量奖励 r（稀疏、延迟）** |
| 数据分布 | 静态 i.i.d. | 静态 | **非 i.i.d.**（自己影响后续数据） |
| 目标 | 泛化误差最小 | 结构发现 | **累积回报最大（长期）** |
| 典型问题 | 分类/回归 | 聚类/降维 | 游戏/控制/推荐/LLM 对齐 |

> **面试金句**：RL 的核心难点是 **信用分配（credit assignment）**——延迟的奖励要回溯到"哪一步动作"；以及 **探索与利用的权衡**。

### 经典应用（背三例）

1. **游戏**：AlphaGo / Atari / Dota2（状态=棋面/画面，动作=落子/操作）
2. **机器人控制**：机械臂抓取、四足行走（状态=关节角速度，动作=力矩）
3. **LLM 对齐**：RLHF（状态=已生成 token 序列，动作=下一个 token，奖励=人类偏好）→ 见 10 篇

---


## 2. MDP 五元组（🔴 必背定义）

### 形式定义

一个马尔可夫决策过程由五元组 $\langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle$ 组成：

- **$\mathcal{S}$（状态集）**：智能体所处的所有可能状态，如网格世界的每个格子
- **$\mathcal{A}$（动作集）**：每个状态下可执行的动作，如 上下左右
- **$\mathcal{P}$（状态转移概率）**：$P(s'|s,a)$ —— 在状态 $s$ 执行动作 $a$ 后转移到 $s'$ 的概率
- **$\mathcal{R}$（奖励函数）**：$R(s,a,s')$ 或 $R(s)$ —— 转移时获得的即时奖励
- **$\gamma \in [0,1]$（折扣因子）**：未来奖励的折现程度

### 马尔可夫性质（🔴）

$$P(s_{t+1}|s_t, a_t, s_{t-1}, a_{t-1}, \dots) = P(s_{t+1}|s_t, a_t)$$

**当前状态已经包含了做决策所需的全部历史信息**——历史被"压缩"进当前状态。这是 MDP 能被贝尔曼方程求解的前提。

### 与马尔可夫链 / 马尔可夫奖励过程的关系（🟡 层级）

```text
马尔可夫链 MC : <S, P>                     （无动作无奖励）
马尔可夫奖励过程 MRP : <S, P, R, γ>        （无动作, 有奖励）
马尔可夫决策过程 MDP : <S, A, P, R, γ>     （有动作, 智能体决策）
```

> **面试追问**：MDP 假设了"状态完全可观测"。真实环境部分可观测时 → **POMDP**（状态被观测/隐状态建模，用信念状态 b(s)）。答出这条直接加分。

---


## 3. 手写 4×4 网格世界（本篇的实验场）

规则设计（经典 Cliff 环境的温柔版）：

- 网格 `4×4=16` 个格子，坐标 `(row, col)`，起点 `(0,0)`，终点 `(3,3)`
- 动作：上/下/左/右 四选一；**撞墙不动**（转移概率演示）
- 移入终点奖励 `+10`，其他正常移动奖励 `-1`（鼓励走最短路径）
- 转移概率：`P(s'|s,a)` 用**确定性移动 + 撞墙自环**模拟（简单但够用于教学）

下面用代码把它表示出来——**状态编号 0~15**，动作编号 0=上 1=下 2=左 3=右。


In [ ]:
# gridworld.py —— 手写 MDP 组件: 状态 / 动作 / 转移 / 奖励（不依赖 gym）
import numpy as np

SIZE = 4
N_STATES = SIZE * SIZE      # 16
N_ACTIONS = 4               # 0上 1下 2左 3右
GOAL = N_STATES - 1         # 15 = (3,3)
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # 上 下 左 右

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def step(s, a):
    """返回 (s_next, reward)。
    确定性版本: 撞墙自环, 到达终点给 +10 并终止(episode 结束)。
    """
    if s == GOAL:
        return GOAL, 0
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return s, -1.0          # 撞墙: 留在原地, 惩罚
    s_next = nr * SIZE + nc
    return (GOAL, 10.0) if s_next == GOAL else (s_next, -1.0)

# 快速自检: 从 0 向下走到 4, 奖励应为 -1
print('状态 (0,0) 向下 →', step(0, 1))
# 从 14 向右走到终点
print('状态 14 向右 →', step(14, 3))
# 从 0 向上撞墙: 自环
print('状态 0 向上(撞墙) →', step(0, 0))
# 底边 12 向下撞墙
print('状态 12 向下(撞墙) →', step(12, 1))


## 4. 回报 Return 与折扣因子 γ（🔴）

### 累积回报定义

从 $t$ 时刻开始的**累积折扣回报**：

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

### γ 的三种含义（背）

1. **数学上**：$\gamma<1$ 保证无限 horizon 的回报**收敛**（$\sum \gamma^k = \frac{1}{1-\gamma}$）
2. **直觉上**：$\gamma$ 小 → 近视（只关心短期）；$\gamma$ 大 → 远视（看重长期）
3. **建模上**：$\gamma=0$ 等价于"贪心只看下一步"；$\gamma\to1$ 等价于"总和回报"

### 边界情况（面试易错）

- $\gamma=0$：$G_t = R_{t+1}$，只看一步（多臂老虎机场景）
- $\gamma=1$ 且任务有限长：回报 = 不加权总和（有限 horizon 收敛）

### 为什么需要折扣（选答）

① 收敛性；② 人类/动物天然偏好即时奖励（延迟满足有限）；③ 避免无限回报发散；④ 处理不确定性（未来奖励置信度低）。

---


In [ ]:
# return_demo.py —— 折扣回报数值演示: 一条轨迹的 G_t 怎么算
import numpy as np

def discounted_return(rewards, gamma, t=0):
    """从 t 时刻起的折扣回报 G_t"""
    return sum(gamma ** k * r for k, r in enumerate(rewards[t:]))

# 示例轨迹奖励: 前面一直 -1, 最后一步 +10
rewards = [-1, -1, -1, -1, 10]

print(f'{"γ":<6}{"G_0":>8}{"G_3":>8}    (G_3 只看最后两步)')
for gamma in [0.0, 0.5, 0.9, 0.99]:
    g0 = discounted_return(rewards, gamma, 0)
    g3 = discounted_return(rewards, gamma, 3)
    print(f'{gamma:<6}{g0:>8.3f}{g3:>8.3f}')

print()
print('γ=0.9 时逐步展开 G_0:')
g = 0.0
for k, r in enumerate(rewards):
    g += 0.9 ** k * r
    print(f'  时刻 {k}: 累加到 {g:.3f}')
print('当 γ→1, G 趋向(+10 永不被深度折扣掩盖), 而 γ 小则后期 +10 几乎无感')


## 5. 状态价值 V(s) 与动作价值 Q(s,a)（🔴 必背定义）

### 状态价值函数（在策略 π 下）

$$V^{\pi}(s) = \mathbb{E}_{\pi}\left[G_t \,\middle|\, S_t = s\right]$$

含义：**从状态 $s$ 出发，按照策略 $\pi$ 行动，能获得的期望累积回报。**

### 动作价值函数

$$Q^{\pi}(s,a) = \mathbb{E}_{\pi}\left[G_t \,\middle|\, S_t = s, A_t = a\right]$$

含义：**在状态 $s$ 先执行动作 $a$，之后按 $\pi$ 行动，的期望累积回报。**

### V 与 Q 的关系（背）

$$V^{\pi}(s) = \sum_{a} \pi(a|s)\, Q^{\pi}(s,a)$$

- $V$ 是对动作按策略概率**加权平均**的 $Q$
- 若策略是确定性的（$\pi(a|s)=1$ 对某 $a$），则 $V^{\pi}(s) = Q^{\pi}(s, a_{\pi})$

### 最优价值函数

$$V^*(s) = \max_{\pi} V^{\pi}(s), \qquad Q^*(s,a) = \max_{\pi} Q^{\pi}(s,a)$$

> **面试金句**：$V$ 回答"这个状态有多好"，$Q$ 回答"这个状态下做这个动作有多好"；$Q$ 比 $V$ 多一个"动作"维度，所以 $Q$ 可以直接指导决策（$a^*=\arg\max_a Q^*(s,a)$），而模型需要知道策略才能用 $V$。

---


## 6. 贝尔曼期望方程（🔴 推导要会）

### 状态价值版

把 $G_t$ 拆成"立即奖励 + 折扣的下一状态价值"：

$$V^{\pi}(s) = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a)\left[ R(s,a,s') + \gamma V^{\pi}(s') \right]$$

### 动作价值版

$$Q^{\pi}(s,a) = \sum_{s'} P(s'|s,a)\left[ R(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^{\pi}(s',a') \right]$$

### 推导三步（背下来，面试让手推就写这个）

```text
① 展开: V(s) = E[ R_{t+1} + γ G_{t+1} | S_t=s ]
② 拆期望: 对 a 求和(策略) × 对 s' 求和(转移概率)
③ 递归: E[G_{t+1}|S_{t+1}=s'] = V(s')
```

### 贝尔曼方程的结构（一句话）

**贝尔曼方程 = 立即奖励 + 折扣 × 下一步价值的期望** —— 它让"未来的价值"变成了"当前价值的线性方程"，于是可以迭代求解。

---


In [ ]:
# bellman_policy_eval.py —— 随机策略下的策略评估: 迭代求解 V^π
# 用贝尔曼期望方程迭代: V_{k+1}(s) = Σ_a π(a|s) Σ_s' P(s'|s,a)[R + γV_k(s')]
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    """确定性转移: 返回 [(概率, s', r)]"""
    if s == N - 1:
        return [(1.0, s, 0.0)]          # 终点吸收
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]         # 撞墙
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

# 随机策略: 每个状态四个动作等概率
pi = np.full((N, 4), 0.25)

V = np.zeros(N)
for it in range(200):
    V_new = np.zeros(N)
    for s in range(N):
        v = 0.0
        for a in range(4):
            for p, s2, r in trans(s, a):
                v += pi[s, a] * p * (r + GAMMA * V[s2])
        V_new[s] = v
    if np.max(np.abs(V_new - V)) < 1e-8:
        V = V_new
        break
    V = V_new

print(f'迭代 {it + 1} 轮后收敛')
print('随机策略下每个状态的价值 V(s)（2 位小数）:')
print(np.round(V.reshape(SIZE, SIZE), 2))
print()
print('解读: 离终点越近价值越高; 角落 (0,0) 因为随机乱走绕远, 价值为负')


## 7. 贝尔曼最优方程（🔴）

### 最优状态价值

$$V^*(s) = \max_{a} \sum_{s'} P(s'|s,a)\left[ R(s,a,s') + \gamma V^*(s') \right]$$

### 最优动作价值

$$Q^*(s,a) = \sum_{s'} P(s'|s,a)\left[ R(s,a,s') + \gamma \max_{a'} Q^*(s',a') \right]$$

### 与期望方程的区别（背）

| | 贝尔曼期望方程 | 贝尔曼最优方程 |
|---|---|---|
| 里面的策略 | 固定策略 π（加权平均） | **max over 动作**（选最优） |
| 解 | V^π（该策略的价值） | V^*（最优价值） |
| 求解 | 策略评估（迭代） | 值迭代 / 策略迭代 |

### 最优策略的性质

- 至少存在一个确定性最优策略：$\pi^*(a|s) = \mathbb{1}\{a = \arg\max_a Q^*(s,a)\}$
- 只要找到 $Q^*$，$\pi^*$ 直接取 argmax 即可——**这就是 Q-learning 的思想来源**

### 两个不动点方程

贝尔曼最优方程是非线性的（因为 max），但它的解 $V^*$ 是唯一不动点；**值迭代**就是对这个不动点做**压缩映射迭代**（Banach 不动点定理保证收敛）。

---


In [ ]:
# value_iteration.py —— 值迭代求解最优 V* 与最优策略（手写, 无调包）
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTIONS = ['↑', '↓', '←', '→']

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

V = np.zeros(N)
for it in range(1000):
    V_new = np.zeros(N)
    for s in range(N):
        best = -np.inf
        for a in range(4):
            v = sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a))
            best = max(best, v)
        V_new[s] = best
    if np.max(np.abs(V_new - V)) < 1e-9:
        V = V_new
        break
    V = V_new

# 由 V* 恢复最优策略: 每格选让 V*(s) 最大的动作
policy = []
for s in range(N):
    vals = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
    policy.append(ACTIONS[int(np.argmax(vals))])

print(f'值迭代 {it + 1} 轮收敛')
print('最优价值 V*:')
print(np.round(V.reshape(SIZE, SIZE), 2))
print()
print('最优策略（每格箭头, * 为终点）:')
for r in range(SIZE):
    row = ''
    for c in range(SIZE):
        s = r * SIZE + c
        row += ('  *' if s == N - 1 else '  ' + policy[s])
    print(row)
print()
print('验证: 从起点沿箭头走 → 路径价值应等于 V*(0,0)')


## 8. 面试高频追问与易错点（🔴）

### 追问速答

1. **Q: γ=0 和 γ=1 分别代表什么？**
   A: γ=0 只看立即奖励（多臂老虎机）；γ=1 看全程总回报且要求有限 horizon。

2. **Q: V(s) 和 Q(s,a) 哪个更有用？**
   A: Q 含动作维度，可直接 argmax 得到策略；V 需要配合策略模型。DQN 家族学 Q，Policy Gradient 家族学策略 π。

3. **Q: 贝尔曼方程为什么能迭代求解？**
   A: 它是关于 V 的线性/非线性不动点方程，满足收缩映射 → Banach 不动点定理保证唯一解与收敛。

4. **Q: MDP 和 POMDP 区别？**
   A: MDP 状态完全可观测；POMDP 只有部分可观测 → 需用信念状态 b(s) 或历史/RNN 近似，难解。

5. **Q: 为什么需要折扣因子（除收敛外）？**
   A: 模型不确定、奖励延迟折价符合直觉、避免无限回报发散。

### 易错点清单

- ❌ 把"回报 G"和"价值 V"搞混：G 是**一条轨迹**的实际累积奖励；V 是**期望**（对所有轨迹平均）
- ❌ 忘记 $V^\pi(s)=\sum_a \pi(a|s)Q^\pi(s,a)$ 中的动作要先**按策略加权**
- ❌ 值迭代里用 $\max$、策略评估里用 $\sum_a \pi(a|s)$——两个方程结构不同，别混
- ❌ 转移概率 $P(s'|s,a)$ 与环境相关、不是智能体学的；智能体学的是**价值/策略**

---

## 9. 自测清单（L1 必会 / L2 进阶）

- [ ] L1：默写 MDP 五元组和马尔可夫性质
- [ ] L1：写出 $V^\pi$、$Q^\pi$、$V^*$、$Q^*$ 四个定义式
- [ ] L1：能解释 V 与 Q 的换算关系 $V^\pi(s)=\sum_a \pi(a|s)Q^\pi(s,a)$
- [ ] L1：手推贝尔曼期望方程（三步：展开/拆期望/递归）
- [ ] L2：说出贝尔曼最优方程为什么非线性、怎么保证收敛
- [ ] L2：能比较策略评估与值迭代的更新式差异
- [ ] L2：现场手写一个 3×3 网格的值迭代（10 分钟内）

>  下一篇 `01-动态规划与表格方法`：策略迭代 / 价值迭代的工程细节 + 收敛性实验。


## 10. 状态转移概率矩阵：把 MDP 的 P 落到实处

### 转移概率的组织方式（背）

$P(s'|s,a)$ 是一个 **3 维表**：对每个状态 $s$、每个动作 $a$，给出一组后继 $s'$ 及概率。表格法的核心数据结构就是这三层字典/张量。

### 确定性 vs 随机转移

- **确定性**：$P(s'|s,a)=1$ 只有一个后继（教学网格常用，本系列 00-05 的默认）
- **随机性**：如"执行动作 80% 按意图走、20% 随机偏转"→ 更接近真实世界（机器人打滑、噪声传感器）

下面把本篇网格的完整转移表打印出来——**先把环境"解剖"清楚，后面所有算法才有基础**。


In [ ]:
# trans_table.py —— 打印完整转移表: 每个(s,a)的后继与奖励
import numpy as np

SIZE, N = 4, 16
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]
GOAL = N - 1

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def successors(s, a):
    """返回 [(s_next, reward)], 确定性转移"""
    if s == GOAL:
        return [(GOAL, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(s, -1.0)]
    s2 = nr * SIZE + nc
    return [(GOAL, 10.0)] if s2 == GOAL else [(s2, -1.0)]

print('状态 s | 动作 a | 后继 s′(prime) | 奖励 r   （部分状态示例）')
print('-' * 48)
for s in [0, 1, 5, 14]:
    for a in range(4):
        for s2, r in successors(s, a):
            print(f'  {s:3d}  |   {a}    |   {s2:3d}   |  {r:5.1f}')
print()
print('撞墙演示: 状态 0(左上角) 执行"上"(a=0) → 自环 0, 奖励 -1')
print('终态 15 是吸收态: 任何动作都留在 15, 奖励 0')


## 11. 马尔可夫性质判定练习（🔴 概念辨析）

### 三问自测

1. **棋盘游戏中只看当前棋面能否做决策？** → 能：棋面包含全部信息 → 马尔可夫
2. **股市只看当前价格做决策？** → 不能：价格历史（趋势/波动率）有信息 → **非马尔可夫**，需把历史特征（MA5/波动率）并入状态
3. **机器人只看当前关节角能否控制？** → 通常不能：还需要**速度**（一阶历史）→ 状态应扩为 (位置, 速度)

### 工程上怎么"马尔可夫化"（背）

把记忆变成状态的一部分：

```text
非马尔可夫:  S_t = 观测 o_t
马尔可夫化:  S_t = (o_t, o_{t-1})      → 一阶记忆
             S_t = (o_t, o_{t-1}, ..., o_{t-k})  → k 阶记忆
             S_t = RNN 隐状态            → 无限记忆近似
```

> **面试金句**：POMDP 的"信念状态"就是马尔可夫化的最一般形式——把所有可能真实状态的概率分布 $b(s)$ 当状态，转移是贝叶斯更新。

---


In [ ]:
# markov_check.py —— 判断给定"状态定义"是否满足马尔可夫性（工程练习）
# 规则: 若下一状态转移只依赖当前状态 → 是马尔可夫

def is_markov(state_desc, needs_history):
    """needs_history=True 表示必须依赖历史才能预测"""
    return not needs_history

cases = [
    ('棋面(ChessBoard)', '预测合法走法', False),        # 棋面含全部信息
    ('仅当前股价', '预测明天涨跌', True),               # 需要趋势
    ('(位置, 速度) 的机器人', '预测下一秒位置', False),  # 牛顿力学一阶马尔可夫
    ('LLM 当前 token 序列', '预测下一个 token', False),  # 序列即历史压缩
]
for state, pred, needs in cases:
    mk = is_markov(state, needs)
    print(f'{state:<22} 预测任务: {pred:<14} → 马尔可夫: {"是" if mk else "否"}')

print()
print('结论: 状态定义得够不够"全"，决定了马尔可夫性是否成立')
print('RL 工程的核心手艺之一: 把状态设计到"当前即可决策"')


## 12. 回报计算手练习题（🔴 计算基本功）

### 题 1

轨迹奖励 $[-1, -1, +10]$，$\gamma=0.9$，求 $G_0$：

$$G_0 = -1 + 0.9\times(-1) + 0.9^2\times 10 = -1 - 0.9 + 8.1 = 6.2$$

### 题 2（陷阱题）

$\gamma=0.5$，同样轨迹：$G_0 = -1 + 0.5\times(-1) + 0.25\times 10 = 1.0$——**注意 γ 小会让后期大奖励大幅缩水**。

### 题 3

无限长常量奖励 $r$：$G = r + \gamma r + \gamma^2 r + \cdots = \dfrac{r}{1-\gamma}$（$\gamma<1$）。

- $r=1, \gamma=0.9$ → $G=10$；$\gamma=0.5$ → $G=2$——**γ 越大"同样奖励流"越值钱**。

---


In [ ]:
# return_math.py —— 三题计算 + 几何级数收敛验证
import numpy as np

def G(rewards, gamma, t=0):
    return sum(gamma ** k * r for k, r in enumerate(rewards[t:]))

print('题1: [-1,-1,+10], γ=0.9 → G0 =', round(G([-1, -1, 10], 0.9), 3), '(手算 6.2)')
print('题2: [-1,-1,+10], γ=0.5 → G0 =', round(G([-1, -1, 10], 0.5), 3), '(手算 1.0)')
print()

# 题3: 无限常量奖励流的收敛: 前 N 项部分和逼近 r/(1-γ)
gamma = 0.9
r = 1.0
true = r / (1 - gamma)
partial = 0.0
for k in range(50):
    partial += gamma ** k * r
print(f'题3: r=1, γ=0.9 → 理论 {true:.3f}, 前50项部分和 {partial:.6f}, 误差 {abs(partial-true):.2e}')
print()
print('验证: γ<1 时部分和以指数速度逼近极限 —— 这就是折扣保证收敛的底层原因')
# 若 γ=1: 部分和线性增长不收敛
print('若 γ=1: 前 100 项和 =', sum(1.0 for _ in range(100)), '(线性增长, 不发散约束则无定义)')


## 13. V 与 Q 的数值互验（🔴 用代码证明公式）

### 要验证的两条关系

$$Q^{\pi}(s,a) = \sum_{s'} P(s'|s,a)\left[ R + \gamma V^{\pi}(s') \right], \qquad V^{\pi}(s) = \sum_a \pi(a|s) Q^{\pi}(s,a)$$

- 由 $V^\pi$ 可推出 $Q^\pi$（第一条）
- 由 $Q^\pi$ 加权可得 $V^\pi$（第二条）
- 两条互相咬合 → 就是贝尔曼方程的两个视角

### 为什么这个关系重要（面试）

> "有了 $V$ 想得到 $Q$"只用一步环境查询（模型已知时）；"有了 $Q$ 想得到 $V$"只用策略加权。**在做 policy evaluation 时它们等价，但 Q 表更耗内存（多一个动作维度）。**

---


In [ ]:
# vq_relation.py —— 数值验证 V-Q 互推关系
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

# 已知 V^π（从 01 篇的策略评估得来; 这里直接迭代到收敛）
pi = np.full((N, 4), 0.25)
V = np.zeros(N)
for _ in range(500):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2]) for a in range(4) for p, s2, r in trans(s, a))
    V = Vn

# ① 由 V 推 Q
Q = np.zeros((N, 4))
for s in range(N):
    for a in range(4):
        Q[s, a] = sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a))

# ② 由 Q 加权推 V
V_rebuilt = (pi * Q).sum(axis=1)

print('关系验证: max|V - ΣπQ| =', float(np.max(np.abs(V - V_rebuilt))))
print('（< 1e-8 即两条关系完全成立）')
print()
print('示例: 状态 0 的四个 Q 值:', np.round(Q[0], 3))
print('      状态 0 的 V^π =', round(V[0], 3), '= 0.25 × ΣQ ✓')


## 14. 贝尔曼方程：迭代 → 线性方程组（🟡 另一种求解视角）

### 两条路

| 解法 | 思路 | 适用 |
|---|---|---|
| **迭代法**（值迭代/策略评估） | 反复套贝尔曼方程直到不动点 | 任意规模, 主流 |
| **线性方程组** | $V = R + \gamma P V \Rightarrow V = (I-\gamma P)^{-1}R$ | 小规模精确解 |

### 解析解数值演示

策略评估在表格小规模下等价于解线性方程组——下面直接求逆得到**精确解**，再与迭代解对比，验证迭代法收敛到真值。

> 面试一般不要求手解线性方程，但要明白"贝尔曼方程是线性方程 → 迭代是 Jacobi 迭代求解线性系统"，数值分析背景会加分。


In [ ]:
# bellman_linsolve.py —— 迭代解 vs 线性方程组精确解对比
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

pi = np.full((N, 4), 0.25)

# 组装: V = R_π + γ P_π V  =>  (I - γP)V = R
P = np.zeros((N, N))
R = np.zeros(N)
for s in range(N):
    for a in range(4):
        for p, s2, r in trans(s, a):
            P[s, s2] += pi[s, a] * p
            R[s] += pi[s, a] * p * r

# 迭代解
V_it = np.zeros(N)
for _ in range(500):
    V_it = R + GAMMA * P @ V_it

# 精确解
V_exact = np.linalg.solve(np.eye(N) - GAMMA * P, R)

print('迭代解 vs 精确解 最大误差:', float(np.max(np.abs(V_it - V_exact))))
print('精确解 V(起点):', round(V_exact[0], 6))
print('迭代解 V(起点):', round(V_it[0], 6))
print()
print('含义: 策略评估 = 解线性方程组; 迭代法(≈Jacobi)对大规模更可行')


## 15. 折扣因子 γ 敏感性实验（🔴 数值直觉）

### 要观察的现象

γ 决定策略的"眼光长短"：

- γ 小（如 0.1）→ 只看近处奖励，可能选"先拿小奖励"的短视路径
- γ 大（如 0.99）→ 看重远期大奖励，愿意绕远路

### 本实验设计

在 4×4 网格（终点 +10、步罚 -1）上对 γ ∈ {0.1, 0.5, 0.9, 0.99} 跑值迭代，观察：起点价值 $V^*(0,0)$ 与最优路径长度。

> **预期**：γ 越大，起点价值越高（因为 +10 被折扣得少），且最优策略越"愿意走弯路"（当 γ 极小甚至无所谓路径，只求尽快结束）。


In [ ]:
# gamma_sweep.py —— γ 扫描: 起点价值与最优策略随 γ 变化
import numpy as np

SIZE = 4
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTIONS = ['↑', '↓', '←', '→']

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a, gamma):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def value_iteration(gamma):
    V = np.zeros(N)
    for _ in range(1000):
        Vn = np.zeros(N)
        for s in range(N):
            best = max(sum(p * (r + gamma * V[s2]) for p, s2, r in trans(s, a, gamma)) for a in range(4))
            Vn[s] = best
        if np.max(np.abs(Vn - V)) < 1e-9:
            V = Vn
            break
        V = Vn
    # 恢复策略并测量路径长度(从0沿贪心箭头走)
    pol = []
    for s in range(N):
        q = [sum(p * (r + gamma * V[s2]) for p, s2, r in trans(s, a, gamma)) for a in range(4)]
        pol.append(int(np.argmax(q)))
    s, path_len = 0, 0
    seen = set()
    while s != N - 1 and s not in seen and path_len < 100:
        seen.add(s)
        s = trans(s, pol[s], gamma)[0][1]   # 沿贪心动作转移(确定性环境唯一后继)
        path_len += 1
    return V[0], path_len

print(f'{"γ":<6}{"V*(0,0)":>12}{"贪心路径步数":>12}')
for gamma in [0.1, 0.5, 0.9, 0.99]:
    v0, plen = value_iteration(gamma)
    print(f'{gamma:<6}{v0:>12.3f}{plen:>12}')
print()
print('观察: γ=0.1 时 V* 最小(远期+10 几乎被抹掉); γ→0.99 时 V* 最大且路径趋于最短路')


## 16. 随机转移 MDP：真正的 P(s'|s,a)（🟡 进阶建模）

### 为什么需要随机转移

真实环境几乎都不是确定性的：机器人轮子打滑（本要向前却滑到左边）、网络丢包重试、游戏里的随机伤害。确定性网格只是教学简化。

### 建模方式

$P(s'|s,a)$ 用"动作 80% 成功 + 20% 向垂直方向偏"来模拟：

```text
执行"上"时: 80% 上移, 10% 左移, 10% 右移 (撞墙则自环)
```

### 随机转移对求解的影响

- 价值/策略必须**按期望**计算——最优策略要"稳健"（选不容易被噪声害惨的动作）
- 值迭代公式不变，只是转移表变密（每个 (s,a) 有多个后继）

---


In [ ]:
# stochastic_grid.py —— 随机转移网格 + 值迭代求解最优策略
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans_stochastic(s, a, p_slip=0.2):
    """p_slip 概率向垂直方向随机偏转"""
    if s == N - 1:
        return [(1.0, s, 0.0)]
    out = []
    perp = [(a + 1) % 2 * 2, a % 2 * 2 + (1 if a % 2 == 0 else -1)]  # 近似垂直的两个方向
    acts = [a] * 3 + [perp[0]] + [perp[1]]          # 80% 直行, 20% 分给垂直
    probs = [0.8] + [0.1, 0.1] if False else [0.8, 0.1, 0.1]
    # 简化: 直行 0.8, 两个垂直候选各 0.1 (若候选不存在则并入自环)
    candidates = [a, (a + 1) % 4, (a + 3) % 4]
    for cand, p in zip(candidates, [0.8, 0.1, 0.1]):
        r, c = divmod(s, SIZE)
        dr, dc = MOVE[cand]
        nr, nc = r + dr, c + dc
        if not in_bounds(nr, nc):
            out.append((p, s, -1.0))                # 撞墙自环
        else:
            s2 = nr * SIZE + nc
            out.append((p, N - 1, 10.0) if s2 == N - 1 else (p, s2, -1.0))
    return out

V = np.zeros(N)
for _ in range(1000):
    Vn = np.zeros(N)
    for s in range(N):
        best = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans_stochastic(s, a)) for a in range(4))
        Vn[s] = best
    if np.max(np.abs(Vn - V)) < 1e-9:
        V = Vn
        break
    V = Vn

ACTIONS = ['↑', '↓', '←', '→']
print('随机转移(直行80%/偏转20%)下的最优价值 V*:')
print(np.round(V.reshape(SIZE, SIZE), 2))
print()
print('与确定性版对比: 角落价值更低(噪声把撞墙概率放大了)')
print('最优策略依然指向终点, 但会避开"贴墙走"的路线(贴墙更容易被噪声撞墙)')


## 17. 价值热力图可视化（🟢 直观理解 V*）

用 matplotlib 把最优价值画成热力图：颜色越亮 = 价值越高。视觉上一眼看出"价值从终点向四周衰减"。

> 本仓库统一自绘 matplotlib 图（外部图片源不可达），中文字体 SimHei/Microsoft YaHei。


In [ ]:
# value_heatmap.py —— V* 热力图 + 最优策略箭头叠加
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

V = np.zeros(N)
for _ in range(1000):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
    if np.max(np.abs(Vn - V)) < 1e-9:
        V = Vn
        break
    V = Vn

# 最优动作箭头方向
arrows = []
for s in range(N):
    q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
    a = int(np.argmax(q))
    arrows.append(MOVE[a])
arrows = np.array(arrows)

fig, ax = plt.subplots(figsize=(4.6, 4.6), dpi=150)
im = ax.imshow(V.reshape(SIZE, SIZE), cmap='viridis', origin='upper')
for r in range(SIZE):
    for c in range(SIZE):
        s = r * SIZE + c
        if s == N - 1:
            ax.text(c, r, '★', ha='center', va='center', fontsize=16, color='white')
            continue
        dr, dc = arrows[s]
        ax.annotate('', xy=(c + dc * 0.32, r + dr * 0.32),
                    xytext=(c - dc * 0.18, r - dr * 0.18),
                    arrowprops=dict(arrowstyle='->', color='white', lw=1.4))
        ax.text(c, r + 0.38, f'{V[s]:.1f}', ha='center', va='center',
                fontsize=7, color='white', alpha=0.9)
ax.set_title('4×4 网格最优价值热力图 + 最优策略箭头', fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('rl_00_heatmap.png', dpi=150)
print('已保存 rl_00_heatmap.png（价值热力图）')
print('终点价值最高, 向角落递减; 箭头全部指向价值上升方向')


## 18. POMDP：状态观测不到的 MDP（🟡 了解概念）

### 定义

部分可观测马尔可夫决策过程 = MDP + **观测模型** $O(o|s',a)$：

- 智能体看不到真实状态 $s$，只能看到观测 $o$
- 例：自动驾驶只有传感器数据（观测），真实"路面状态"不可见

### 解法核心（背一句话）

用**信念状态** $b(s) = P(S_t = s | 历史)$ 替代真实状态，转移 = 贝叶斯更新：

$$b'(s') \propto O(o|s',a) \sum_s P(s'|s,a) b(s)$$

### 为什么难

信念空间是**连续概率分布**（无限维）→ 精确求解 PSPACE-hard；工程上用近似（粒子滤波/DRQN/RNN 隐状态）。

> **面试金句**："RL 处理部分可观测的标准套路是 ①把状态定义成历史特征 ②用 RNN 隐状态 ③用信念状态做规划"。

---


## 19. 状态爆炸：为什么大问题不用表格 DP（🔴 规模直觉）

### 组合爆炸数字（背）

| 场景 | 状态数 | 表格 V 表大小 |
|---|---|---|
| 4×4 网格 | 16 | 微不足道 |
| Atari 画面 | $256^{84\times84}$ ≈ $10^{17000}$ | 宇宙原子数 $10^{80}$ 的 $10^{16920}$ 倍 |
| 围棋 | $10^{170}$ | 完全存不下 |
| 连续控制 | 无限（实数） | 不可能 |

### 结论（背）

- 表格法只适用于**小规模有限**状态（教学/玩具/小棋盘）
- 大规模 → **函数近似**：$V_\theta(s)$ 用神经网络参数化 → 泛化到未见状态 → **深度 RL**（04 篇）

---


In [ ]:
# state_explosion.py —— 状态爆炸数字演示
import math

def log10_space(grid_size):
    """粗略估: 每像素 256 灰度, 状态数 = 256^(像素数)"""
    return grid_size * grid_size * math.log10(256)

for grid in [4, 21, 84, 210]:
    e = log10_space(grid)
    print(f'Atari 画面 {grid}x{grid}: 状态数 ≈ 10^{e:.0f}  (宇宙原子 ≈ 10^80)')
print()
print('结论: 16 个格子能画表格, 21x21 已不可能, 84x84 完全是天文数字')
print('→ 深度 RL 用神经网络把"输入画面 → 价值"泛化出来, 而不是枚举状态')


## 20. 本章易错点补充（🔴）

1. **回报是轨迹量、价值是期望量**：G 是"这一条走出来的"，V 是"平均意义上"——面试画图题常考
2. **V→Q 要查环境**（用 P/R），**Q→V 只要策略加权**——别把两者来源搞混
3. **折扣收敛 vs 无限回报**：γ<1 才保证无限 horizon 收敛；γ=1 只对有限任务有定义
4. **马尔可夫 ≠ 无记忆**：是"状态已含足够记忆"，不是"不需要记忆"
5. **随机策略 vs 确定性策略**：表格法两者都支持；随机策略对博弈环境（石头剪刀布）必要
6. **POMDP 不是"加了噪声的 MDP"**：是观测缺失，解法是信念状态而不是加大状态

---

## 21. 本章面试追问扩展（🟡 进阶 8 问）

1. **Q: γ 调大会有什么副作用？** A: 价值更长远但收敛更慢、方差更大；无限 horizon 需 γ<1。
2. **Q: 值迭代每轮复杂度？** A: O(|S|²|A|)（每个 (s,a) 遍历所有后继），实际用稀疏转移优化。
3. **Q: 状态转移概率未知怎么用 V→Q？** A: 采样估计：$Q(s,a)\approx \frac{1}{N}\sum [r + \gamma V(s')]$（用经验平均）。
4. **Q: 为什么叫"模型"？** A: P 和 R 就是环境模型；model-based = 用它们做规划，model-free = 不学它们。
5. **Q: 回报 G 的方差为什么大？** A: 轨迹随机性（随机策略 + 随机转移）全部累积进 G。
6. **Q: 半马尔可夫（SMDP）？** A: 动作持续随机时长才结束（时间抽象）；选项（options）框架。
7. **Q: 状态聚合/抽象？** A: 把相似状态合并成抽象状态降维（如 tile coding）；是表格到函数近似的桥梁。
8. **Q: 多步回报 G_{t:t+n}？** A: n 步截断回报 = n 步真实奖励 + 第 n+1 步价值估计；MC 与 TD 的统一视角。

## 22. 自测清单（L3 冲刺）

- [ ] L3：能现场推出 $Q^\pi$ 与 $V^\pi$ 的互推公式
- [ ] L3：解释"策略评估 = 解线性方程组 (I-γP)V=R"并说明迭代法对应 Jacobi
- [ ] L3：画出状态爆炸量级对比（棋盘/Atari/围棋）
- [ ] L3：说出 POMDP 的信念状态更新式
- [ ] L3：γ 敏感性实验的预期结论随口能讲


## 23. 世界模型视角：为什么"模型"这么重要（🟡 理论升华）

### 奖励/转移的两层含义

- **学规划**：有 P/R 直接 DP（01 篇）；没有就采样学（02 篇）
- **学世界模型**：从数据学 $P_\theta(s'|s,a)$、$R_\theta$ → 用学到的模型做**想象训练**（Dyna-Q / Dreamer / MuZero）

### 面试一句话

> "RL 的所有算法本质都在回答同一件事：**如何在没有完美模型的约束下逼近贝尔曼最优**——价值法用估计价值、策略法用采样梯度、MBRL 先学模型再规划。"

---


In [ ]:
# bellman_contraction.py —— 贝尔曼算子的收缩性数值演示(Banach 不动点)
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def bellman_op(V):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
    return Vn

# 取两个初始价值, 迭代一步看距离变化: ||T(V1)-T(V2)|| <= γ·||V1-V2||
rng = np.random.RandomState(0)
V1 = rng.rand(N) * 10
V2 = rng.rand(N) * 10
for step in range(1, 4):
    d_before = np.max(np.abs(V1 - V2))
    V1, V2 = bellman_op(V1), bellman_op(V2)
    d_after = np.max(np.abs(V1 - V2))
    print(f'第{step}步: 距离 {d_before:.3f} -> {d_after:.3f}  (收缩系数≤γ={GAMMA})')

print()
print('→ 贝尔曼最优算子 T 是 γ-收缩映射 → 迭代必收敛到唯一 V*')


## 24. 决策视角：V 与 Q 的"分层"理解（🔴 面试表达）

### 三种"决策深度"

| 问题 | 主语 | 需要什么 |
|---|---|---|
| 这个状态好不好？ | $V(s)$ | 状态价值 |
| 这个状态下做这个动作？ | $Q(s,a)$ | 动作价值 |
| 接下来该做什么？ | $\pi(s)$ | 策略 |

### 面试常见陷阱

- 问"V 与 Q 谁大？" → $\max_a Q(s,a) \ge V(s)$（最优时取等）；$V(s)=\sum_a\pi(a|s)Q(s,a)$ 在**任意策略下**成立（不是只有最优）
- 问"策略迭代里为什么用 Q 改进？" → 改进用 $\arg\max_a Q^{\pi}(s,a)$，Q 里带着动作维度

---


In [ ]:
# vq_inequality.py —— 数值验证: max_a Q(s,a) >= V(s)
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

# 随机策略的 V 与 Q
pi = np.full((N, 4), 0.25)
V = np.zeros(N)
for _ in range(400):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                    for a in range(4) for p, s2, r in trans(s, a))
    V = Vn

Q = np.zeros((N, 4))
for s in range(N):
    for a in range(4):
        Q[s, a] = sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a))

ok = all(np.max(Q[s]) >= V[s] - 1e-9 for s in range(N))
print(f'所有状态满足 max_a Q(s,a) >= V(s) ? {ok}')
print('差值示例(状态0): maxQ-V =', round(np.max(Q[0]) - V[0], 4))
print('→ 价值与动作价值的不等式: 按更好动作走至少不比平均差')


## 25. 本章终极易错点（🔴 考前 10 分钟看）

1. **γ=0 时 V(s)=E[R] 与价值无关——多臂老虎机问题**
2. **回报 G 与累计折扣 Σγ^k r 是一回事**——别在计算里重复折扣
3. **贝尔曼方程适用任意策略**（期望式）；只有**最优方程**里是 max
4. **状态转移概率是对"环境"的建模，不是策略**——策略是智能体选择动作的规则
5. **V* 唯一、最优策略不唯一**（多个动作同样最优）
6. **表格 vs 函数近似**的分水岭：能否枚举所有状态

---

## 26. 自测清单（00 篇终版）

- [ ] L1-L3 全部旧条目可背写
- [ ] 新增：能解释贝尔曼算子收缩性的一句话证明
- [ ] 新增：随手画一个 3 状态 MDP 并手算一步 V 更新
- [ ] 新增：说出"为什么要折扣"的 4 个理由


## 27. 本章公式总表（🔴 考前速背）

| 概念 | 公式 |
|---|---|
| 回报 | $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$ |
| 状态价值 | $V^{\pi}(s)=\mathbb{E}_\pi[G_t\mid S_t=s]$ |
| 动作价值 | $Q^{\pi}(s,a)=\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a]$ |
| V↔Q | $V^{\pi}(s)=\sum_a \pi(a\mid s)Q^{\pi}(s,a)$ |
| 贝尔曼期望 | $V^{\pi}(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)[R+\gamma V^{\pi}(s')]$ |
| 贝尔曼最优 | $V^*(s)=\max_a\sum_{s'}P(s'\mid s,a)[R+\gamma V^*(s')]$ |
| 最优策略 | $\pi^*(a\mid s)=\mathbb{1}\{a=\arg\max_a Q^*(s,a)\}$ |

> 考前把这张表默写 3 遍，MDP 篇就过关了。


## 28. 与最优控制/强化学习家族的关系图（🟢 知识面）

```text
动态规划(Bellman) ── 连续版 → HJB 方程(最优控制)
        │
        ├── 有模型表格: 策略/值迭代(01)
        ├── 无模型表格: MC/TD/Q-learning(02/03)
        └── 函数近似:   DQN/PPO/SAC(04-06)
强化学习 = 动态规划 + 蒙特卡洛 + 函数近似 三线合流
```

> 面试若问"RL 的数学根基"，答：**马尔可夫 + 贝尔曼最优性 + 大数定律（采样估计）**，三句话收尾。


In [ ]:
# mdp_manual_exercise.py —— 手动小练习: 3 状态 MDP 一步 V 更新(带答案校验)
import numpy as np

# 3 状态 MDP: 状态2 是吸收(奖励0); 状态0/1 动作0->状态1(奖励1), 动作1->状态2(奖励5)
# V 初始为 [0,0,0], 计算一步策略评估(确定性策略: 全部动作0)
GAMMA = 0.9
R = {0: 1.0, 1: 1.0, 2: 0.0}
next_s = {0: 1, 1: 2, 2: 2}

V = np.zeros(3)
for step in range(3):
    Vn = np.array([R[s] + GAMMA * V[next_s[s]] for s in range(3)])
    print(f'第{step+1}步: V = {np.round(Vn, 3)}')
    V = Vn

print()
print('手算: V0(1) = 1 + 0.9*V1(0) = 1; V0(2) = 1+0.9*1 = 1.9; V0(3)=1+0.9*1.9=2.71')
print('→ 与程序一致 → 贝尔曼一步更新的手感建立')


## 29. 回看：从"环境"到"求解"的完整闭环（🔴 串讲）

```text
环境(4×4网格) → MDP 五元组(S,A,P,R,γ) → 价值函数(V/Q) → 贝尔曼方程
  → 迭代求解(值迭代/策略评估) → 最优策略 π* → 用 ε-greedy 部署
```

> 本篇完成后你应该能：给出任意网格→写出 MDP→手算 V/Q→跑值迭代→画出最优路径。这一整条流水线是 RL 岗位面试的第一问。


## 30. 变种模型谱系：MRP / MDP / POMDP / BMDP（🔴 推导关系）

### 三步演进（背推导链）

```text
MRP  : <S, P, R, γ>                    无动作, 只做预测
MDP  : <S, A, P, R, γ>   MRP + 动作   智能体可决策
BMDP : <S, A, Q>         已知 Q 表     查表即最优
POMDP: <S, A, O, P, R, γ>  部分可观    状态不可见, 用信念状态
```

### 关键推导：MDP → 策略诱导的 MRP

给定策略 $\pi$，把"状态-动作"揉进状态转移：

$$P^{\pi}(s' \mid s) = \sum_a \pi(a \mid s) P(s' \mid s, a), \qquad R^{\pi}(s) = \sum_a \pi(a \mid s) R(s, a)$$

> **推导一步（背）**：有了 $\langle S, P^{\pi}, R^{\pi}, \gamma \rangle$，这就是一个 MRP——所以 $V^{\pi}$ 满足 MRP 的贝尔曼方程。**"策略的价值 = 该策略诱导出的 MRP 的价值"**是整条谱系的枢纽。


## 31. 贝尔曼方程逐步推导（🔴 面试手推模板）

### 出发定义

$$V^{\pi}(s) = \mathbb{E}_{\pi}\left[ G_t \mid S_t = s \right], \qquad G_t = R_{t+1} + \gamma G_{t+1}$$

### 四步展开（背）

```text
① 代入回报递归:  V(s) = E[ R_{t+1} + γ·G_{t+1} | S_t = s ]
② 拆期望:       = E[ R_{t+1} | s ] + γ·E[ G_{t+1} | s ]
                    a           a'
③ 对内层期望按动作展开（策略 π）:
       = Σ_a π(a|s) R(s,a) + γ·Σ_a π(a|s) Σ_s' P(s'|s,a) · E[G_{t+1}|s']
④ 识别 V(s'):   = Σ_a π(a|s) [ R(s,a) + γ Σ_s' P(s'|s,a) V(s') ]   ∎
```

> **易错点**：第 ③ 步的 $E[G_{t+1}|s']$ 在马尔可夫性下等于 $V(s')$——这一步去掉条件 $S_t=s$ 是"马尔可夫性"的直接使用，面试官会盯着这里问。


In [ ]:
# fig_discount.py —— 过程图①: 折扣回报的权重衰减(不同 γ)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
import os

steps = np.arange(0, 20)
fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=150)
for gamma, c in [(0.0, 'gray'), (0.5, 'orange'), (0.9, 'green'), (0.99, 'blue')]:
    ax.plot(steps, gamma ** steps, 'o-', ms=4, label=f'γ={gamma}', color=c)
ax.set_xlabel('未来时刻 t'); ax.set_ylabel('γ^t（奖励权重）')
ax.set_title('折扣因子 γ 对远期奖励权重的衰减（推导 G_t 的几何权重）')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
import os as _os
out = _os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if _os.path.basename(_os.getcwd()) == '教学':
    out = _os.path.join(_os.getcwd(), 'images')
_os.makedirs(out, exist_ok=True)
fig.savefig(_os.path.join(out, 'bellman_discount.png'), dpi=150)
print('已保存 images/bellman_discount.png')
plt.close(fig)
plt.show()


In [ ]:
# fig_contraction.py —— 过程图②: 贝尔曼算子收缩(不同初值收敛到同一 V*)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_b(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_b(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def bellman_op(V):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a))
                    for a in range(4))
    return Vn

rng = np.random.RandomState(0)
V_a, V_b = rng.rand(N) * 10, np.zeros(N)
trace_a, trace_b = [], []
for _ in range(40):
    trace_a.append(V_a[0]); trace_b.append(V_b[0])
    V_a, V_b = bellman_op(V_a), bellman_op(V_b)

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=150)
ax.plot(trace_a, 'o-', ms=3, label='初值A(随机)', color='purple')
ax.plot(trace_b, 's-', ms=3, label='初值B(全0)', color='teal')
ax.set_xlabel('迭代轮数 k'); ax.set_ylabel('V_k(状态0)')
ax.set_title('贝尔曼最优算子是 γ-收缩映射：不同初值收敛到同一 V*')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'bellman_contraction.png'), dpi=150)
print('已保存 images/bellman_contraction.png')
plt.close(fig)
plt.show()


## 📸 本章推导配图（教学/images/）

![折扣因子 γ 对远期奖励权重的衰减（推导 G_t 的几何权重）](images/bellman_discount.png)

> 折扣因子 γ 对远期奖励权重的衰减（推导 G_t 的几何权重）

![贝尔曼最优算子是 γ-收缩映射：不同初值收敛到同一 V*](images/bellman_contraction.png)

> 贝尔曼最优算子是 γ-收缩映射：不同初值收敛到同一 V*
